In [ ]:
import logging
import sys
from pathlib import Path
sys.path.append('src')
import utils

logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - [%(name)s] - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

logger = logging.getLogger(__name__)
logger.info("Módulos importados e logging configurado.")

In [ ]:
# --- Parâmetros de Configuração ---

# Arquivos e diretórios
GRID_SHAPEFILE = 'grid/grid_bh.shp'
OUTPUT_X_PATH = "data/X/"
OUTPUT_Y_PATH = "data/y/"
CHECKPOINT_FILE = "processing_checkpoint.json"

# Parâmetros do Catálogo STAC
CATALOG_URL = "http://aqui.io/cuborizonte/catalogo/stac"
TARGET_CRS = "EPSG:31983"
RESOLUTION = 0.8 # em metros

# Configuração dos dados de entrada (X)
X_CONFIG = {
    "product_name": "bh_ortophoto",
    "bands": ["red", "green", "blue"],
    "time_interval": "1999-01-01/1999-12-31",
    "output_path": Path(OUTPUT_X_PATH)
}

# Configuração dos dados de alvo (y)
Y_CONFIG = {
    "product_name": "bh_lidar_rasterized",
    "bands": ["classification"],
    "time_interval": "2015-01-01/2015-12-31",
    "output_path": Path(OUTPUT_Y_PATH)
}

logger.info("Parâmetros de configuração definidos.")

In [ ]:
X_CONFIG["output_path"].mkdir(parents=True, exist_ok=True)
Y_CONFIG["output_path"].mkdir(parents=True, exist_ok=True)
logger.info(f"Diretórios de saída '{OUTPUT_X_PATH}' e '{OUTPUT_Y_PATH}' garantidos.")

bbox_list = utils.create_bbox_list_from_shapefile(GRID_SHAPEFILE)

if not bbox_list:
    logger.warning("A lista de bboxes está vazia. O processamento será encerrado.")

In [ ]:
# Carrega o estado do checkpoint
checkpoint = utils.load_checkpoint(CHECKPOINT_FILE)
completed = set(checkpoint['completed'])
failed = set(checkpoint['failed'])

logger.info(f"Iniciando processamento para {len(bbox_list)} grids.")
logger.info(f"{len(completed)} grids já concluídos. {len(failed)} falharam anteriormente.")

# Itera sobre cada grid para processamento
for grid_name, grid_bbox in bbox_list:
    if grid_name in completed:
        logger.info(f"Pulando grid '{grid_name}' - já processado com sucesso.")
        continue
    
    logger.info(f"--- Processando grid: {grid_name} ---")

    try:
        # --- Processamento dos dados X (Ortofoto) ---
        x_file = X_CONFIG["output_path"] / f"{grid_name}.tif"
        x_success = False
        if x_file.exists():
            logger.info(f"Arquivo X já existe para '{grid_name}'.")
            x_success = True
        else:
            logger.info(f"Buscando dados X para '{grid_name}' de {X_CONFIG['time_interval']}")
            x_datacube = utils.get_datacube(
                CATALOG_URL, grid_bbox, X_CONFIG['product_name'], X_CONFIG['time_interval'],
                RESOLUTION, X_CONFIG['bands'], TARGET_CRS
            )
            x_success = utils.process_and_save_datacube(x_datacube, x_file)

        # --- Processamento dos dados Y (LIDAR) ---
        y_file = Y_CONFIG["output_path"] / f"{grid_name}.tif"
        y_success = False
        if y_file.exists():
            logger.info(f"Arquivo Y já existe para '{grid_name}'.")
            y_success = True
        else:
            logger.info(f"Buscando dados Y para '{grid_name}' de {Y_CONFIG['time_interval']}")
            y_datacube = utils.get_datacube(
                CATALOG_URL, grid_bbox, Y_CONFIG['product_name'], Y_CONFIG['time_interval'],
                RESOLUTION, Y_CONFIG['bands'], TARGET_CRS
            )
            y_success = utils.process_and_save_datacube(y_datacube, y_file)

        # --- Atualiza o checkpoint ---
        # Consideramos sucesso se AMBOS os arquivos (X e Y) foram gerados
        if x_success and y_success:
            completed.add(grid_name)
            failed.discard(grid_name) # Remove da lista de falhas se teve sucesso agora
            logger.info(f"SUCESSO no processamento do grid '{grid_name}'.")
        else:
            failed.add(grid_name)
            logger.warning(f"FALHA no processamento do grid '{grid_name}'. Verifique os logs.")

    except Exception as e:
        logger.error(f"Erro crítico não esperado ao processar o grid {grid_name}: {e}", exc_info=True)
        failed.add(grid_name)

    finally:
        # Salva o checkpoint a cada iteração para não perder progresso
        utils.save_checkpoint(CHECKPOINT_FILE, {
            'completed': list(completed),
            'failed': list(failed)
        })

logger.info("--- Processamento concluído. ---")
logger.info(f"Total de grids com sucesso: {len(completed)}")
logger.info(f"Total de grids com falha: {len(failed)}")